# cTreeBalls versus ENCORE: survey 2PCF and 3PCF

This notebook exercises `search=octree-3pcf-3d-omp` in its ENCORE-style survey mode. It creates a data catalog and a random catalog in a non-trivial survey window, constructs the normalized $D-\alpha R$ and $\alpha R$ catalogs, and compares cTreeBalls with ENCORE at three levels:

1. raw 2PCF pair counts,
2. raw $N_\ell$ and $R_\ell$ 3PCF multipoles,
3. the final survey-window-corrected 2PCF and 3PCF.

The highest measured multipole is used as the extra edge-correction order. Thus a run with `measured_lmax=3` reports science multipoles through $\ell=2$.

## Requirements

- `cyballs` compiled with `OCTREE3PCF3DOMPON = 1`, or set `backend="cli"` and provide a matching `cballs` executable.
- NumPy and Matplotlib.
- A C++ compiler and the ENCORE source directory. The helper builds a native CPU-only ENCORE executable without modifying that source tree.

Set the `ENCORE_SOURCE` environment variable when ENCORE is not in the default local path.

In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd() / 'examples', Path.cwd()]
EXAMPLES_DIR = next(
    path for path in candidates
    if (path / 'compare_octree_3pcf_3d_encore.py').is_file()
)
PROJECT_ROOT = EXAMPLES_DIR.parent
sys.path.insert(0, str(EXAMPLES_DIR))

from compare_octree_3pcf_3d_encore import (
    ComparisonConfig, generate_catalogs, print_summary, run_comparison,
)

In [ ]:
ENCORE_SOURCE = Path(os.environ.get(
    'ENCORE_SOURCE',
    '/Users/mar/Documents/Codex/lyman_alpha/Eladio/encore_2026-08-27',
))
OUTPUT_DIR = Path(os.environ.get(
    'CTREEBALLS_ENCORE_OUTPUT',
    EXAMPLES_DIR / 'octree_3pcf_3d_encore_notebook_output',
))

config = ComparisonConfig(
    output_dir=OUTPUT_DIR,
    encore_source=ENCORE_SOURCE,
    cballs_executable=PROJECT_ROOT / 'cballs',
    backend=os.environ.get('CTREEBALLS_EXAMPLE_BACKEND', 'cython'),
    n_data=90,
    n_random=300,
    nbins=6,
    measured_lmax=3,
    rmin=0.05,
    rmax=0.80,
    threads=4,
    seed=8675309,
)
config

## Inspect the synthetic survey

The random catalog samples the survey selection. The data catalog uses the same selection but includes two clustered components, giving a visible correlation signal. Both catalogs carry positive, position-dependent weights.

In [ ]:
import matplotlib.pyplot as plt

data_pos, data_w, random_pos, random_w = generate_catalogs(config)
fig, ax = plt.subplots(figsize=(7.5, 6.0))
ax.scatter(random_pos[:, 0], random_pos[:, 1], s=9, alpha=0.30, label='Random')
ax.scatter(data_pos[:, 0], data_pos[:, 1], s=22, alpha=0.85, label='Data')
ax.set(xlabel='x', ylabel='y', title='Synthetic survey catalog')
ax.set_aspect('equal')
ax.grid(alpha=0.2)
ax.legend();

## Run both estimators

For cTreeBalls, the Cython path passes both catalogs from NumPy memory using `set_catalog(..., catalog=0/1)`. The helper then runs ENCORE twice: once on $D-\alpha R$ and once on positive $\alpha R$. ENCORE's supplied coupling tensor is used to reproduce its edge-correction solve.

In [ ]:
result = run_comparison(config)
print_summary(result['summary'])

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUTPUT_DIR / 'comparison_2pcf.png')))
display(Image(filename=str(OUTPUT_DIR / 'comparison_3pcf.png')))

## Inspect numerical differences

`comparison_2pcf.csv` and `comparison_3pcf.csv` contain the two estimates, absolute differences, relative differences, and raw numerator/random counts. ENCORE writes raw counts with `%le`, so agreement below roughly $10^{-6}$ relative precision is hidden by text-output rounding.

In [ ]:
import csv

with (OUTPUT_DIR / 'comparison_3pcf.csv').open() as stream:
    rows = list(csv.DictReader(stream))

for row in rows[:8]:
    print(
        f"ell={row['ell']} bins=({row['bin1']},{row['bin2']}) "
        f"cTreeBalls={float(row['ctreeballs_zeta_encore_basis']): .6e} "
        f"ENCORE={float(row['encore_zeta']): .6e} "
        f"rel={float(row['relative_difference']): .3e}"
    )

## Output conventions

The comparison uses cTreeBalls' `zeta_encore` column, not `zeta_legendre`, because ENCORE reports the basis $(-1)^\ell\sqrt{2\ell+1}P_\ell/(4\pi)$. The raw comparison likewise uses cTreeBalls' `N_encore` and `R_encore` columns. The separate Legendre-basis coefficient remains available in `histZetaM_3d_survey.txt`.